In [2]:
!nvidia-smi

Tue Oct 27 19:11:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
print("Installing Unsloth and dependencies...")

Installing Unsloth and dependencies...


In [4]:
# Install Unsloth and memory-optimization libraries
print("Installing Unsloth and dependencies...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
print("Environment setup complete!")

Installing Unsloth and dependencies...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ra85eg1k/unsloth_43af31f216104bcaaade84547de904ef
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ra85eg1k/unsloth_43af31f216104bcaaade84547de904ef
  Resolved https://github.com/unslothai/unsloth.git to commit c4f9de9c058094ba420701a64df6f3c63291807d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 66.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 123.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 59.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 148.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 143.

In [5]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Maximum text length the model can handle at once
dtype = None # Auto-detection for our T4 GPU
load_in_4bit = True # Use 4-bit quantization (QLoRA) to shrink the model so it fits in free VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct", # The elite open-source coder model we are building upon
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Leobrain base model loaded successfully into memory!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Leobrain base model loaded successfully into memory!


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for speed
    bias = "none",    # Optimized to "none" for speed
    use_gradient_checkpointing = "unsloth", # Saves VRAM drastically!
    random_state = 3407,
)

print("LoRA adapters attached! Leobrain is primed for training.")

Unsloth 2026.10.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
LoRA adapters attached! Leobrain is primed for training.


In [8]:
import json
from datasets import Dataset

# 1. Sample dataset tailored for Leobrain
raw_data = [
    {
        "instruction": "Write a Python function to check if a string is a palindrome.",
        "input": "text = 'racecar'",
        "output": "def is_palindrome(s):\n    cleaned = ''.join(c.lower() for c in s if c.isalnum())\n    return cleaned == cleaned[::-1]"
    },
    {
        "instruction": "Write a secure database connection snippet using Python and SQLite.",
        "input": "",
        "output": "import sqlite3\n\ndef create_connection(db_file):\n    try:\n        conn = sqlite3.connect(db_file)\n        return conn\n    except sqlite3.Error as e:\n        print(e)\n    return None"
    },
    {
        "instruction": "Explain how to handle division by zero safely in Python.",
        "input": "",
        "output": "def safe_divide(a, b):\n    try:\n        return a / b\n    except ZeroDivisionError:\n        return 'Error: Cannot divide by zero.'"
    }
]

# 2. Prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

# FIXED: Returns a dictionary containing a list of strings (required for batched=True)
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }

# 3. Convert and map
dataset = Dataset.from_list(raw_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

print("Dataset created and formatted successfully!")

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset created and formatted successfully!


In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30, # Small number of steps for a quick initial test run
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Start the training process!
print("Starting Leobrain training loop...")
trainer_stats = trainer.train()
print("Training complete! Leobrain has successfully learned from the dataset.")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting Leobrain training loop...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss
1,1.733322
2,1.733322
3,1.704873
4,1.590196
5,1.361161
6,1.050361
7,0.831489
8,0.707254
9,0.604242
10,0.513172


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


Training complete! Leobrain has successfully learned from the dataset.


In [10]:
from unsloth import FastLanguageModel

# Set model to inference mode for fast generation
FastLanguageModel.for_inference(model)

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Give Leobrain a test coding prompt
prompt = prompt_template.format(
    "Write a Python function to check if a string is a palindrome.",
    "text = 'developer'",
    ""
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True
)

response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a Python function to check if a string is a palindrome.

### Input:
text = 'developer'

### Response:
def is_palindrome(s):
    cleaned = ''.join(c.lower() for c in s if c.isalnum())
    return cleaned == cleaned[::-1]


In [11]:
# 1. Save your fine-tuned Leobrain adapters locally in Colab
model.save_pretrained("leobrain_adapter")
tokenizer.save_pretrained("leobrain_adapter")
print("Leobrain weights saved successfully!")

# 2. Install FastAPI, Uvicorn, and Pyngrok for hosting
!pip install fastapi uvicorn pyngrok nest-asyncio

import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from threading import Thread

# Enable nested loops for Colab
nest_asyncio.apply()

app = FastAPI(title="Leobrain Online API")

class PromptRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 256

@app.post("/generate")
def generate_code(request: PromptRequest):
    try:
        inputs = tokenizer([request.prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=request.max_new_tokens, use_cache=True)
        response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        return {"status": "success", "response": response}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 3. Start the FastAPI server in a background thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server running locally on port 8000...")

Unsloth: Restored added_tokens_decoder metadata in leobrain_adapter/tokenizer_config.json.


Leobrain weights saved successfully!
FastAPI server running locally on port 8000...


In [14]:
from pyngrok import ngrok

# Paste your actual token inside these quotes
ngrok.set_auth_token("3LI6OtySai02xMi83o2mB0hZpaC_3hThn6TLGj4w6UXoA76Dn")

# Open a public tunnel to port 8000
public_url = ngrok.connect(8000)
print(f"==================================================")
print(f"🚀 LEOBRAIN IS NOW LIVE ONLINE!")
print(f"Public API URL: {public_url}")
print(f"==================================================")

🚀 LEOBRAIN IS NOW LIVE ONLINE!
Public API URL: NgrokTunnel: "https://stonework-unleaded-justly.ngrok-free.dev" -> "http://localhost:8000"
